  # Whisper Accent Robustness — Model Performance Evaluation







  Run `eval_model_perf.py` on SLURM first to generate the CSVs this notebook reads.







  - **Combined** → scripted + spontaneous test data (single CSV per model)

  - **Per-domain** → filter by `domain` column within each CSV







  Metrics:



  - **WER**  — word error rate (primary ASR metric)

  - **PER**  — phoneme error rate via G2P; labelled "PER (G2P)" throughout

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
RESULTS_DIR = "results/model_perf_comparison"


# Keys must match {model_key} in CSV filenames; values are display labels
MODEL_KEYS = {
    "baseline":          "Zero-shot",
    # "no_aux_heldout_chinese": "Naive LoRA FT (h/o Chinese)",
    "whisper_ft":            "Naive LoRA FT",
    # "no_aux_heldout_chinese":        "Naive LoRA FT (heldout Chinese)",
    # "ctc_aux_l3":        "CTC Aux",
    # "feat_aux":          "Feat Aux",
    # "feat_aux_heldout_chinese":      "Feat Aux (heldout Chinese)",
    # "feat_aux0p3":       "Feat Aux (0.3)",
    # "both_aux":          "CTC + Feat",

    "whisfusion": "WhisFusion (Zero-shot)",
    "whisfusion_ft": "WhisFusion (LoRA FT)",
    "whisfusion_ft_hoc": "WhisFusion (LoRA FT, heldout Chinese)",
}
DOMAINS = ["scripted", "spontaneous"]  # Filter options within combined CSVs




In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from jiwer import wer as jiwer_wer

pd.set_option("display.max_colwidth", 80)




In [ ]:
def load_results(model_key: str) -> pd.DataFrame | None:
    """Load combined predictions CSV (scripted + spontaneous)."""
    p = Path(RESULTS_DIR) / f"{model_key}_predictions.csv"
    if not p.exists():
        print(f"  [missing] {p} — run eval_model_perf.py first")
        return None
    return pd.read_csv(p)


def filter_by_domain(df: pd.DataFrame, domain: str | None) -> pd.DataFrame:
    """Filter combined DataFrame by domain (or return all)."""
    if domain is None:
        return df
    return df[df["domain"] == domain].copy()


# ── Load all cached CSVs ──────────────────────────────────────────────────────
results: dict[str, pd.DataFrame | None] = {}
for key in MODEL_KEYS:
    df = load_results(key)
    results[key] = df
    if df is not None:
        print(f"  loaded  {key}: {len(df):,} rows ({df['domain'].value_counts().to_dict()})")
    else:
        print(f"  [missing] {key}")



In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────


def available(key: str) -> bool:
    return results.get(key) is not None


def get_data(key: str, domain: str | None = None) -> pd.DataFrame | None:
    """Get data for model, optionally filtered by domain."""
    df = results.get(key)
    if df is None:
        return None
    return filter_by_domain(df, domain)


def corpus_wer(df: pd.DataFrame) -> float:
    return float(jiwer_wer(
        df["reference_norm"].fillna("").tolist(),
        df["prediction_norm"].fillna("").tolist(),
    ))


def corpus_per(df: pd.DataFrame) -> float | None:
    """Mean utterance PER (G2P-derived)."""
    if "utt_per" not in df.columns:
        return None
    vals = df["utt_per"].dropna()
    return float(vals.mean()) if len(vals) else None


def grouped_wer(df: pd.DataFrame, group_col: str = "l1") -> pd.DataFrame:
    rows = []
    for grp, sub in df.groupby(group_col):
        rows.append({
            group_col:  grp,
            "num_utts": len(sub),
            "wer":      float(jiwer_wer(
                                sub["reference_norm"].fillna("").tolist(),
                                sub["prediction_norm"].fillna("").tolist(),
                            )),
            "per":      float(sub["utt_per"].dropna().mean())
                        if "utt_per" in sub.columns else None,
        })
    return pd.DataFrame(rows)


print("Helpers loaded.")



  ---



  # Part 1 — Overall WER & PER (G2P)



  **Default: Combined (all data).** Use `domain="scripted"` or `"spontaneous"` in `get_summary_table()` for domain-specific views.

In [ ]:
def get_summary_table(domain: str | None = None) -> pd.DataFrame:
    """WER/PER table for domain (None = all data)."""
    rows = []
    for key, label in MODEL_KEYS.items():
        df = get_data(key, domain)
        if df is None:
            continue
        wer = corpus_wer(df)
        per = corpus_per(df)
        domain_str = f" ({domain})" if domain else " (All Data)"
        rows.append({"Model": f"{label}{domain_str}", "WER": wer, "PER (G2P)": per})
    return pd.DataFrame(rows)


domain = None
df = get_summary_table(domain)
display(
    df.style
    .format("{:.4f}")
    .background_gradient(cmap="RdYlGn_r")
    .set_caption("WER & PER (G2P) — All Data (Scripted + Spontaneous)")
)


In [ ]:
from plotly.subplots import make_subplots


def plot_summary_bars(domain: str | None = None):
    """Plot WER/PER bars for domain (None = all data)."""
    sub = get_summary_table(domain)
    if sub.empty:
        return
    
    domain_str = f" ({domain})" if domain else " (All Data)"
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("WER", "PER (G2P)"),
        shared_yaxes=True,
    )
    
    # WER
    wer_vals = sub["WER"]
    fig.add_trace(
        go.Bar(
            name="WER",
            x=sub["Model"].tolist(),
            y=wer_vals.tolist(),
            text=[f"{v:.1%}" if pd.notna(v) else "" for v in wer_vals],
            textposition="outside",
            showlegend=False,
        ),
        row=1, col=1
    )
    
    # PER
    per_vals = sub["PER (G2P)"]
    if per_vals.notna().any():
        fig.add_trace(
            go.Bar(
                name="PER (G2P)",
                x=sub["Model"].tolist(),
                y=per_vals.tolist(),
                text=[f"{v:.1%}" if pd.notna(v) else "" for v in per_vals],
                textposition="outside",
                showlegend=False,
            ),
            row=1, col=2
        )
    
    fig.update_yaxes(title_text="Error Rate", tickformat=".0%", row=1, col=1)
    fig.update_yaxes(tickformat=".0%", row=1, col=2)
    fig.update_layout(
        title=f"WER vs PER by Model{domain_str}",
        margin=dict(t=90),
    )
    fig.show()

domain = None
# ── Combined plot (default) ───────────────────────────────────────────────────
plot_summary_bars(domain)


  ---



  # Part 2 — Per-L1 WER



  **Default: Combined (all data).** Use `domain=` for domain-specific L1 breakdowns.

In [ ]:
def get_l1_breakdown(domain: str | None = None) -> pd.DataFrame:
    """Per-L1 WER breakdown."""
    l1_rows = []
    for key, label in MODEL_KEYS.items():
        df = get_data(key, domain)
        if df is None:
            continue
        grp = grouped_wer(df, "l1")
        grp["Model"] = label
        l1_rows.append(grp)
    
    if not l1_rows:
        return pd.DataFrame()
    
    l1_df = pd.concat(l1_rows, ignore_index=True)
    
    # Delta vs zero-shot baseline (negative = improvement)
    if "baseline" in MODEL_KEYS and available("baseline"):
        base_label = MODEL_KEYS["baseline"]
        base_wer = (
            l1_df[l1_df["Model"] == base_label][["l1", "wer"]]
            .rename(columns={"wer": "wer_base"})
        )
        l1_df = l1_df.merge(base_wer, on="l1", how="left")
        l1_df["wer_delta_pct"] = (
            (l1_df["wer"] - l1_df["wer_base"]) / l1_df["wer_base"] * 100
        )
    
    return l1_df


# ── Combined L1 breakdown (default) ───────────────────────────────────────────
domain = None
l1_df = get_l1_breakdown(domain)
if not l1_df.empty:
    l1s = sorted(l1_df["l1"].unique())
    
    # Explicit model order
    model_order = [
        "baseline",
        "no_aux",
        "no_aux_heldout_chinese",
        "whisfusion",
        "whisfusion_ft",
        "whisfusion_ft_hoc",
    ]
    
    ordered_items = [
        (k, MODEL_KEYS[k])
        for k in model_order
        if k in MODEL_KEYS and available(k)
    ]
    
    fig = go.Figure()
    for key, label in ordered_items:
        sub = l1_df[l1_df["Model"] == label].set_index("l1")
        fig.add_trace(go.Bar(
            name=label,
            x=l1s,
            y=[sub.loc[l, "wer"] if l in sub.index else None for l in l1s],
            text=[f"{sub.loc[l, 'wer']:.1%}" if l in sub.index else "" for l in l1s],
            textposition="outside",
        ))
    
    fig.update_layout(
        title="WER by L1 — All Data (Scripted + Spontaneous)",
        barmode="group",
        yaxis=dict(title="WER", tickformat=".0%"),
        legend=dict(orientation="h", y=1.12, xanchor="center", x=0.5),
        margin=dict(t=80),
    )
    fig.show()
    
    out = Path(RESULTS_DIR) / "comparison_all_by_l1.csv"
    l1_df.to_csv(out, index=False)
    print(f"Saved {out}")
    display(
        l1_df.sort_values(["l1", "Model"])
             .style.format({
                 "wer":           "{:.4f}",
                 "wer_base":      "{:.4f}",
                 "wer_delta_pct": "{:+.1f}%",
                 "per":           lambda v: f"{v:.4f}" if pd.notna(v) else "—",
             })
             .set_caption("All Data — Per-L1 WER")
    )



  ---



  # Part 3 — Domain Gap Analysis (zero-shot)



  Scripted vs spontaneous **within** each L1, using baseline model.

In [ ]:
def plot_domain_gap(model_key: str = "baseline"):
    """Scripted vs spontaneous gap by L1."""
    df = results.get(model_key)
    if df is None:
        return
    
    s_g  = grouped_wer(df[df["domain"] == "scripted"], "l1").rename(columns={"wer": "WER_scripted"})
    sp_g = grouped_wer(df[df["domain"] == "spontaneous"], "l1").rename(columns={"wer": "WER_spontaneous"})
    
    gap = s_g[["l1", "WER_scripted"]].merge(
        sp_g[["l1", "WER_spontaneous"]], on="l1", how="inner")
    gap["gap"] = gap["WER_spontaneous"] - gap["WER_scripted"]
    
    if gap.empty:
        print(f"No domain gap data for {model_key}")
        return
    
    l1s = gap["l1"].tolist()
    fig = go.Figure()
    fig.add_trace(go.Bar(name="Scripted", x=l1s, y=gap["WER_scripted"].tolist()))
    fig.add_trace(go.Bar(name="Spontaneous", x=l1s, y=gap["WER_spontaneous"].tolist()))
    fig.update_layout(
        title=f"{MODEL_KEYS.get(model_key, model_key)} — Scripted vs Spontaneous by L1",
        barmode="group",
        yaxis=dict(title="WER", tickformat=".0%"),
        legend=dict(orientation="h", y=1.12, xanchor="center", x=0.5),
    )
    fig.show()
    display(
        gap.style
         .format({c: "{:.4f}" for c in ["WER_scripted", "WER_spontaneous", "gap"]})
         .set_caption(f"{MODEL_KEYS.get(model_key, model_key)} — Domain Gap by L1")
    )


plot_domain_gap("baseline")



  ---



  # Part 4 — Utterance-level Analysis



  **Default: Combined**. Filter by `domain=` for specific views.

In [ ]:
# ── UTT WER distributions ─────────────────────────────────────────────────────
def plot_utt_wer_dist(domain: str | None = None):
    """Utterance WER histogram."""
    traces = []
    for key in MODEL_KEYS:
        df = get_data(key, domain)
        if df is None:
            continue
        traces.append(go.Histogram(
            x=df["utt_wer"],
            name=MODEL_KEYS[key],
            opacity=0.5,
            nbinsx=40,
        ))
    
    if not traces:
        return
    
    domain_str = f" ({domain})" if domain else " (All Data)"
    fig = go.Figure()
    fig.add_traces(traces)
    fig.update_layout(
        title=f"Utterance WER Distribution{domain_str}",
        barmode="overlay",
        xaxis=dict(title="Utterance WER"),
        yaxis=dict(title="Count"),
        legend=dict(orientation="h", y=1.12, xanchor="center", x=0.5),
    )
    fig.show()

domain = None
plot_utt_wer_dist(domain)


  ---



  # Part 5 — Worst Utterances (cross-model)



  **Default: Combined**. Use `domain=` for filtering.

In [ ]:
# ── Worst utterances per model — cross-model comparison ───────────────────────
N_WORST = 30
models = [
    "whisper_ft",
    "whisfusion_ft",
]
domain = None  # None = all data; "scripted"; "spontaneous"


base_key = models[0]
base_df = get_data(base_key, domain)


for anchor_key in models:
    if not available(anchor_key):
        continue
    
    anchor_df = get_data(anchor_key, domain)
    idx = anchor_df.nlargest(N_WORST, "utt_wer").index
    
    worst = base_df.loc[idx, ["utterance_id", "speaker", "l1", "domain", "reference_norm"]].copy()
    
    # Add each model's prediction + WER + PER for these utterances
    for key in models:
        if not available(key):
            continue
        other = get_data(key, domain)
        col = key
        worst[f"pred_{col}"] = other.loc[idx, "prediction_norm"].values
        worst[f"wer_{col}"]  = other.loc[idx, "utt_wer"].values
        if "utt_per" in other.columns:
            worst[f"per_{col}"] = other.loc[idx, "utt_per"].values
    
    domain_str = f" (domain={domain})" if domain else " (All Data)"
    fmt_cols = {c: "{:.3f}" for c in worst.columns if c.startswith(("wer_", "per_"))}
    display(
        worst.style
             .format(fmt_cols)
             .set_caption(f"Top-{N_WORST} Worst Utterances for {MODEL_KEYS[anchor_key]}{domain_str}")
    )



  ---



  # Part 6 — Held-out L1 Performance



  **Default: Combined**. Use `domain=` for domain-specific.

In [ ]:
def plot_heldout_l1(held_out_l1: str = "English", domain: str | None = None):
    """Held-out L1 comparison."""
    outdir = Path("output")
    outdir.mkdir(exist_ok=True)
    
    rows = []
    for key in MODEL_KEYS:
        df = get_data(key, domain)
        if df is None or "l1" not in df.columns:
            continue
        
        sub = df[df["l1"] == held_out_l1]
        if len(sub) == 0:
            continue
        
        row = {
            "model": MODEL_KEYS[key],
            "n_utts": len(sub),
            "n_speakers": sub["speaker"].nunique() if "speaker" in sub.columns else None,
            "wer_mean": corpus_wer(sub) * 100,
        }
        
        if "utt_per" in sub.columns:
            row["per_mean"] = sub["utt_per"].mean()
            row["per_std"] = sub["utt_per"].std()
        
        rows.append(row)
    
    if not rows:
        return
    
    heldout_cmp = pd.DataFrame(rows).reset_index(drop=True)
    suffix = f"_{domain}" if domain else "_combined"
    heldout_cmp.to_csv(outdir / f"heldout_l1_{held_out_l1}{suffix}.csv", index=False)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    models = heldout_cmp["model"].tolist()
    wers = heldout_cmp["wer_mean"].tolist()
    ax.bar(models, wers, color="#01696f", alpha=0.7)
    ax.set_ylabel("WER (%)")
    ax.set_title(f"Held-out {held_out_l1} — {'All Data' if domain is None else domain.capitalize()} — WER mean")
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.savefig(outdir / f"heldout_l1_{held_out_l1}{suffix}.png", dpi=200, bbox_inches="tight")
    plt.show()
    
    print(f"Saved {outdir / f'heldout_l1_{held_out_l1}{suffix}.csv'}")
    display(heldout_cmp)


# Default: combined
plot_heldout_l1("English")


# Per-domain
for domain in DOMAINS:
    plot_heldout_l1("English", domain)